In [ ]:
!apt-get install libhdf5-dev
!pip install h5py laspy numpy
!pip install open3d matplotlib

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
libhdf5-dev is already the newest version (1.10.7+repack-4ubuntu2).
0 upgraded, 0 newly installed, 0 to remove and 34 not upgraded.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.3/84.3 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.7/447.7 MB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 105.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.7/101.7 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.0/228.0 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 89.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 58.6 MB/s eta 0:00:00
  Attempting uninstall: widgetsnbextension
    Found existing installation: widgetsnbextension 3.6.10
    Uninstalling widgetsnbextension-3

In [ ]:
import torch
print(torch.__version__)  # Check current PyTorch version
print(torch.cuda.is_available())  # Check if GPU is available

2.6.0+cu124
True


In [ ]:
import pandas as pd
import os
import h5py
import numpy as np
import laspy
import glob # Needed to find files
import warnings
import matplotlib.pyplot as plt # For colormap and display
import open3d as o3d
import time
import plotly.graph_objects as go
import argparse
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Path to dataset
DATASET_ROOT = "/content/drive/My Drive/FOR-Instance_Dataset"
METADATA_FILE = os.path.join(DATASET_ROOT, "data_split_metadata.csv")

# Read CSV metadata
df = pd.read_csv(METADATA_FILE)

# Filter only 'dev' files
dev_files = df[df["split"] == "dev"]["path"].tolist()
test_files = df[df["split"] == "test"]["path"].tolist()  # Add test files

In [ ]:
# Split 80% train, 20% validation
train_files, val_files = train_test_split(dev_files, test_size=0.2, random_state=42)

print(f"Training files: {len(train_files)}")
print(f"Validation files: {len(val_files)}")

Training files: 44
Validation files: 12


In [ ]:
# --- Helper Function to save a SMALLER, PRE-FILTERED block (chunk) ---
# <<< PREPROCESSING CHANGE: Renamed and modified function >>>
def save_filtered_hdf5_chunk(data, labels, tree_ids, filename):
    """Saves a chunk of pre-filtered data to an HDF5 file."""
    # Basic check (data should already be filtered, so just check non-empty)
    if data is None or data.shape[0] == 0:
         print(f"⚠ Warning: Attempted to save an empty chunk to {filename}. Skipping.")
         return False
    if not (data.shape[0] == labels.shape[0] == tree_ids.shape[0]):
         print(f"❌ Error: Mismatched shapes before saving chunk to {filename} - "
               f"Data: {data.shape}, Labels: {labels.shape}, Tree IDs: {tree_ids.shape}. Skipping.")
         return False

    try:
        print(f"💾 Saving chunk: Data {data.shape} to {filename}")
        with h5py.File(filename, 'w') as f:
            # Store data, label, treeID as separate datasets
            f.create_dataset('data', data=data, compression="gzip")
            f.create_dataset('label', data=labels, compression="gzip")
            f.create_dataset('treeID', data=tree_ids, compression="gzip")
        return True
    except Exception as e:
        print(f"❌ Error saving chunk to {filename}: {e}")
        return False

In [ ]:
# <<< PREPROCESSING CHANGE: Renamed and modified function >>>
def convert_las_to_filtered_hdf5_chunks(file_list, base_data_dir, output_subdir, output_prefix,
                                        target_points_per_chunk, unclassified_label_id=0):
    """
    Convert LAS files to multiple smaller, pre-filtered HDF5 files (chunks).

    Args:
        file_list (list): List of relative paths to LAS files.
        base_data_dir (str): The root directory where LAS files are located.
        output_subdir (str): The subdirectory name within base_data_dir to save HDF5 chunks.
        output_prefix (str): Prefix for the output HDF5 chunk filenames (e.g., 'train', 'val').
        target_points_per_chunk (int): Target number of *valid* points for each HDF5 chunk file.
        unclassified_label_id (int): The label ID to filter out.
    """
    output_dir = os.path.join(base_data_dir, output_subdir)
    os.makedirs(output_dir, exist_ok=True)
    print(f"Output directory for chunks: {output_dir}")

    # Accumulators for the current chunk (stores VALID points)
    current_chunk_data = []
    current_chunk_labels = []
    current_chunk_tree_ids = []
    points_in_current_chunk = 0
    chunk_index = 0
    files_processed = 0
    total_valid_points_processed = 0

    for file_path in file_list:
        las_file = os.path.join(base_data_dir, file_path)

        if not os.path.exists(las_file) or not las_file.endswith(".las"):
            print(f"⚠ Warning: File not found or not a .las file: {las_file}. Skipping.")
            continue

        try:
            las = laspy.read(las_file)
            files_processed += 1

            if las.header.point_count == 0:
                 print(f"⚠ Warning: Empty LAS file: {las_file}. Skipping.")
                 continue

            # Extract data
            xyz = np.vstack((las.x, las.y, las.z)).T
            # <<< PREPROCESSING CHANGE: Add color extraction if available/needed >>>
            # Example: Assuming RGB are stored as uint16 scaled 0-65535
            # Normalize color to 0-1 range (adjust scale if needed)
            # If colors are 0-255 uint8, just divide by 255.0
            red = las.red / 65535.0 if 'red' in las.point_format.dimension_names else np.zeros(las.header.point_count)
            green = las.green / 65535.0 if 'green' in las.point_format.dimension_names else np.zeros(las.header.point_count)
            blue = las.blue / 65535.0 if 'blue' in las.point_format.dimension_names else np.zeros(las.header.point_count)
            # Combine XYZ and Color (assuming your model wants 6 features)
            # Adjust feature order/content as needed by your model!
            points_data = np.hstack((xyz, red[:,None], green[:,None], blue[:,None])).astype(np.float32)

            labels = np.array(las.classification).astype(np.uint8) # Ensure correct dtype

            if 'treeID' in las.point_format.dimension_names:
                tree_ids = np.array(las['treeID']).astype(np.int32) # Ensure correct dtype
            else:
                print(f"⚠ Warning: 'treeID' not found in {file_path}. Using zeros.")
                tree_ids = np.zeros_like(labels, dtype=np.int32)

            # Sanity check shapes
            if not (points_data.shape[0] == labels.shape[0] == tree_ids.shape[0]):
                 print(f"❌ Error: Mismatched shapes in {file_path}. Skipping file.")
                 continue

            # <<< PREPROCESSING CHANGE: Filter out unclassified points >>>
            valid_mask = (labels != unclassified_label_id)
            points_filtered = points_data[valid_mask, :]
            labels_filtered = labels[valid_mask]
            tree_ids_filtered = tree_ids[valid_mask]

            num_valid_points_in_las = points_filtered.shape[0]
            total_valid_points_processed += num_valid_points_in_las

            if num_valid_points_in_las == 0:
                print(f"ℹ️ Info: No valid points found in {file_path} after filtering. Skipping.")
                continue

            # <<< PREPROCESSING CHANGE: Add filtered points to accumulators >>>
            # Process the filtered points, potentially splitting across multiple chunks
            points_added_to_chunks = 0
            while points_added_to_chunks < num_valid_points_in_las:
                points_needed_for_current_chunk = target_points_per_chunk - points_in_current_chunk
                points_available_from_las = num_valid_points_in_las - points_added_to_chunks
                points_to_add = min(points_needed_for_current_chunk, points_available_from_las)

                start_idx = points_added_to_chunks
                end_idx = points_added_to_chunks + points_to_add

                current_chunk_data.append(points_filtered[start_idx:end_idx, :])
                current_chunk_labels.append(labels_filtered[start_idx:end_idx])
                current_chunk_tree_ids.append(tree_ids_filtered[start_idx:end_idx])
                points_in_current_chunk += points_to_add
                points_added_to_chunks += points_to_add

                # If the current chunk is full, save it
                if points_in_current_chunk >= target_points_per_chunk:
                    # Concatenate data for this chunk
                    chunk_data_np = np.concatenate(current_chunk_data, axis=0)
                    chunk_labels_np = np.concatenate(current_chunk_labels, axis=0)
                    chunk_tree_ids_np = np.concatenate(current_chunk_tree_ids, axis=0)

                    h5_filename = os.path.join(output_dir, f"{output_prefix}_chunk_{chunk_index}.h5")
                    if save_filtered_hdf5_chunk(chunk_data_np, chunk_labels_np, chunk_tree_ids_np, h5_filename):
                         chunk_index += 1
                    else:
                         print(f"Failed to save chunk {chunk_index}. Check errors above.")

                    # Reset accumulators for the next chunk
                    current_chunk_data = []
                    current_chunk_labels = []
                    current_chunk_tree_ids = []
                    points_in_current_chunk = 0

        except Exception as e:
            print(f"❌ Error processing file {las_file}: {e}")

    # --- After the loop, save any remaining points as the last chunk ---
    if points_in_current_chunk > 0:
        print(f"Saving remaining {points_in_current_chunk} valid points as final chunk...")
        chunk_data_np = np.concatenate(current_chunk_data, axis=0)
        chunk_labels_np = np.concatenate(current_chunk_labels, axis=0)
        chunk_tree_ids_np = np.concatenate(current_chunk_tree_ids, axis=0)
        h5_filename = os.path.join(output_dir, f"{output_prefix}_chunk_{chunk_index}.h5")
        if save_filtered_hdf5_chunk(chunk_data_np, chunk_labels_np, chunk_tree_ids_np, h5_filename):
             chunk_index += 1
        else:
             print(f"Failed to save final chunk {chunk_index}.")

    print(f"✅ Finished conversion for '{output_prefix}'. Processed {files_processed} LAS files.")
    print(f"   Total valid points processed: {total_valid_points_processed}")
    print(f"   Created {chunk_index} HDF5 chunk files.")

In [ ]:
def convert_las_to_filtered_hdf5_chunks_by_folder(file_list, base_data_dir, output_base_subdir, output_prefix,
                                           target_points_per_chunk, unclassified_label_id=0):
    """
    Convert LAS files to multiple smaller, pre-filtered HDF5 files (chunks),
    saving them into a nested structure: [output_base_subdir]/[SourceFolder]/[output_prefix]_chunks/[filename.h5]

    Args:
        file_list (list): List of relative paths to LAS files (e.g., ['CULS/site1.las', 'NIBIO/site2.las']).
        base_data_dir (str): The root directory where LAS files are located (e.g., DATASET_ROOT).
        output_base_subdir (str): The base subdirectory name within base_data_dir to save HDF5 chunks
                                  for this split (e.g., 'preprocessed/by_folder/train_chunks').
        output_prefix (str): Prefix for the output HDF5 chunk filenames (e.g., 'train', 'val', 'test').
                             This prefix is also used to create the nested directory level.
        target_points_per_chunk (int): Target number of *valid* points for each HDF5 chunk file.
        unclassified_label_id (int): The label ID to filter out.
    """
    # output_root_dir will be something like:
    # /content/drive/My Drive/FOR-Instance_Dataset/preprocessed_folders/train_chunks
    # /content/drive/My Drive/FOR-Instance_Dataset/preprocessed_folders/val_chunks
    # /content/drive/My Drive/FOR-Instance_Dataset/preprocessed_folders/test_chunks
    output_root_dir = os.path.join(base_data_dir, output_base_subdir)
    os.makedirs(output_root_dir, exist_ok=True)
    print(f"Base output directory for '{output_prefix}' chunks (By Folder - Nested): {output_root_dir}")

    files_processed_count = 0
    total_valid_points_processed = 0
    chunks_saved_count = 0

    # Dictionary to keep track of the next chunk index for each source folder/split combination
    # This ensures unique chunk indices within each final nested folder (e.g., CULS/test_chunks)
    chunk_indices_per_nested_folder = {}

    print(f"Starting conversion for {len(file_list)} files...")

    for i, file_path in enumerate(file_list):
        full_las_path = os.path.join(base_data_dir, file_path)
        source_folder = os.path.dirname(file_path) # Get the source folder name (e.g., 'CULS')

        # --- MODIFICATION HERE ---
        # Define the specific output directory for this source folder *and* split,
        # adding the extra level like 'test_chunks' inside the source folder directory.
        # The final path for files will be:
        # output_root_dir / SourceFolder / [output_prefix]_chunks / filename.h5
        # Example: /.../preprocessed_folders/test_chunks/CULS/test_chunks/test_CULS_chunk_0.h5
        file_output_dir = os.path.join(output_root_dir, source_folder, f"{output_prefix}_chunks") # <-- Added f"{output_prefix}_chunks"
        os.makedirs(file_output_dir, exist_ok=True) # Create this new, deeper subdirectory

        if not os.path.exists(full_las_path) or not full_las_path.endswith(".las"):
            print(f"⚠ Warning: File not found or not a .las file: {full_las_path}. Skipping.")
            continue

        # print(f"\nProcessing file {i+1}/{len(file_list)}: {file_path} -> Saving to {file_output_dir}") # Too verbose

        try:
            las = laspy.read(full_las_path)
            files_processed_count += 1

            if las.header.point_count == 0:
                 print(f"⚠ Warning: Empty LAS file: {full_las_path}. Skipping.")
                 continue

            # Extract data - XYZ, and Color if available
            # Ensure data types are consistent and handled correctly
            xyz = np.vstack((las.x, las.y, las.z)).T.astype(np.float32)
            red = las.red.astype(np.float32) / 65535.0 if 'red' in las.point_format.dimension_names else np.zeros(las.header.point_count, dtype=np.float32)
            green = las.green.astype(np.float32) / 65535.0 if 'green' in las.point_format.dimension_names else np.zeros(las.header.point_count, dtype=np.float32)
            blue = las.blue.astype(np.float32) / 65535.0 if 'blue' in las.point_format.dimension_names else np.zeros(las.header.point_count, dtype=np.float32)
            # Combine XYZ and Color (assuming 6 features)
            points_data = np.hstack((xyz, red[:,None], green[:,None], blue[:,None])).astype(np.float32)

            labels = np.array(las.classification).astype(np.uint8) # Ensure correct dtype

            if 'treeID' in las.point_format.dimension_names:
                tree_ids = np.array(las['treeID']).astype(np.int32) # Ensure correct dtype
            else:
                # print(f"⚠ Warning: 'treeID' not found in {file_path}. Using zeros.") # Too verbose, keep only if needed for debugging
                tree_ids = np.zeros_like(labels, dtype=np.int32)

            # Sanity check shapes
            if not (points_data.shape[0] == labels.shape[0] == tree_ids.shape[0]):
                 print(f"❌ Error: Mismatched shapes in {full_las_path} ({points_data.shape[0]}, {labels.shape[0]}, {tree_ids.shape[0]}). Skipping file.")
                 continue

            # Filter out unclassified points
            valid_mask = (labels != unclassified_label_id)
            points_filtered = points_data[valid_mask, :]
            labels_filtered = labels[valid_mask]
            tree_ids_filtered = tree_ids[valid_mask]

            num_valid_points_in_las = points_filtered.shape[0]
            total_valid_points_processed += num_valid_points_in_las

            if num_valid_points_in_las == 0:
                # print(f"  ℹ️ Info: No valid points found in {os.path.basename(file_path)} after filtering. Skipping.") # Too verbose
                continue

            # print(f"  Found {num_valid_points_in_las} valid points in {os.path.basename(file_path)}.") # Too verbose

            # Split filtered points from THIS file into chunks and save
            points_added_to_chunks_from_this_file = 0

            # Use the full nested folder path as the key for chunk indexing
            nested_folder_key = file_output_dir
            current_nested_folder_chunk_index = chunk_indices_per_nested_folder.get(nested_folder_key, 0)

            while points_added_to_chunks_from_this_file < num_valid_points_in_las:
                start_idx = points_added_to_chunks_from_this_file
                # Determine end_idx
                end_idx = min(start_idx + target_points_per_chunk, num_valid_points_in_las)

                chunk_data_np = points_filtered[start_idx:end_idx, :]
                chunk_labels_np = labels_filtered[start_idx:end_idx]
                chunk_tree_ids_np = tree_ids_filtered[start_idx:end_idx]

                # Construct filename including split prefix, source folder and a unique index for this NESTED folder
                # Example: 'test_CULS_chunk_0.h5' (filename prefix is still useful)
                # The base directory now ensures the correct nested structure.
                h5_filename = os.path.join(file_output_dir, f"{output_prefix}_{source_folder}_chunk_{current_nested_folder_chunk_index}.h5")

                # print(f"    Saving chunk {current_nested_folder_chunk_index}: Points {chunk_data_np.shape[0]} to {os.path.basename(h5_filename)}") # Too verbose
                if save_filtered_hdf5_chunk(chunk_data_np, chunk_labels_np, chunk_tree_ids_np, h5_filename):
                    chunks_saved_count += 1
                    current_nested_folder_chunk_index += 1 # Increment index only on successful save

                points_added_to_chunks_from_this_file = end_idx # Move to the end of the saved slice

            # Update the dictionary with the next available index for this nested folder
            chunk_indices_per_nested_folder[nested_folder_key] = current_nested_folder_chunk_index


        except Exception as e:
            print(f"❌ Error processing file {full_las_path}: {e}")
            # Optional: print traceback for more details
            # import traceback
            # traceback.print_exc()


    print(f"\n✅ Finished conversion for '{output_prefix}' (By Folder - Nested Output).")
    print(f"   Total LAS files processed: {files_processed_count}")
    print(f"   Total valid points processed (across all files): {total_valid_points_processed}")
    print(f"   Total HDF5 chunks saved: {chunks_saved_count}")
    print(f"   Chunk distribution by nested folder:\n   {chunk_indices_per_nested_folder}")

In [ ]:
DATASET_ROOT = "/content/drive/My Drive/FOR-Instance_Dataset"
OUTPUT_ROOT = "/content/drive/My Drive/FOR-Instance_Dataset/preprocessed_chunks"

METADATA_FILE = os.path.join(DATASET_ROOT, "data_split_metadata.csv")

# --- Configuration ---
UNCLASSIFIED_LABEL_ID = 0
# <<< PREPROCESSING CHANGE: Define target size for *each small chunk file* >>>
# Adjust this based on desired file size vs. number of files
# Start with 50k or 100k points per chunk. Too small = too many files. Too large = slow load in getitem.
POINTS_PER_HDF5_CHUNK = 100000 # Example: 100k points per chunk file

# --- Read Metadata and Split Files ---
print("Reading metadata...")
try:
    df = pd.read_csv(METADATA_FILE)
except FileNotFoundError:
    print(f"❌ Error: Metadata file not found at {METADATA_FILE}")
    exit()

# Assuming 'path' column contains paths relative to DATASET_ROOT
dev_files = df[df["split"] == "dev"]["path"].tolist()
test_files = df[df["split"] == "test"]["path"].tolist()

if not dev_files:
     print("⚠ Warning: No 'dev' files found in metadata.")
     train_files, val_files = [], []
else:
     train_files, val_files = train_test_split(dev_files, test_size=0.2, random_state=42)

print(f"Total LAS files found: Train: {len(train_files)}, Validation: {len(val_files)}, Test: {len(test_files)}")
print(f"Target points per HDF5 chunk file: {POINTS_PER_HDF5_CHUNK}")
print(f"Unclassified label ID to filter: {UNCLASSIFIED_LABEL_ID}")
print(f"Output directory for chunks: {OUTPUT_ROOT}")

# --- Define Output Subdirectories within OUTPUT_ROOT ---
# <<< PREPROCESSING CHANGE: Use OUTPUT_ROOT >>>
TRAIN_CHUNK_SUBDIR = "train_chunks"
VAL_CHUNK_SUBDIR = "val_chunks"
TEST_CHUNK_SUBDIR = "test_chunks"

# --- Convert the datasets ---
print("\n--- Converting Training Set ---")
convert_las_to_filtered_hdf5_chunks(train_files, DATASET_ROOT, os.path.join(OUTPUT_ROOT, TRAIN_CHUNK_SUBDIR), "train", POINTS_PER_HDF5_CHUNK, UNCLASSIFIED_LABEL_ID)

print("\n--- Converting Validation Set ---")
convert_las_to_filtered_hdf5_chunks(val_files, DATASET_ROOT, os.path.join(OUTPUT_ROOT, VAL_CHUNK_SUBDIR), "val", POINTS_PER_HDF5_CHUNK, UNCLASSIFIED_LABEL_ID)

print("\n--- Converting Test Set ---")
convert_las_to_filtered_hdf5_chunks(test_files, DATASET_ROOT, os.path.join(OUTPUT_ROOT, TEST_CHUNK_SUBDIR), "test", POINTS_PER_HDF5_CHUNK, UNCLASSIFIED_LABEL_ID)

print("\n🏁 All conversions attempted.")

Reading metadata...
Total LAS files found: Train: 44, Validation: 12, Test: 26
Target points per HDF5 chunk file: 100000
Unclassified label ID to filter: 0
Output directory for chunks: /content/drive/My Drive/FOR-Instance_Dataset/preprocessed_chunks

--- Converting Training Set ---
Output directory for chunks: /content/drive/My Drive/FOR-Instance_Dataset/preprocessed_chunks/train_chunks
⚠ Warning: File not found or not a .las file: /content/drive/My Drive/FOR-Instance_Dataset/NIBIO2/plot41_annotated.las. Skipping.
⚠ Warning: File not found or not a .las file: /content/drive/My Drive/FOR-Instance_Dataset/NIBIO2/plot33_annotated.las. Skipping.
💾 Saving chunk: Data (100000, 6) to /content/drive/My Drive/FOR-Instance_Dataset/preprocessed_chunks/train_chunks/train_chunk_0.h5
💾 Saving chunk: Data (100000, 6) to /content/drive/My Drive/FOR-Instance_Dataset/preprocessed_chunks/train_chunks/train_chunk_1.h5
💾 Saving chunk: Data (100000, 6) to /content/drive/My Drive/FOR-Instance_Dataset/preproc

In [ ]:
DATASET_ROOT = "/content/drive/My Drive/FOR-Instance_Dataset"
# Point OUTPUT_ROOT to the level that will contain train_chunks, val_chunks, test_chunks
OUTPUT_ROOT = "/content/drive/My Drive/FOR-Instance_Dataset/preprocessed_folders" # This seems correct

METADATA_FILE = os.path.join(DATASET_ROOT, "data_split_metadata.csv")

# --- Configuration ---
UNCLASSIFIED_LABEL_ID = 0
POINTS_PER_HDF5_CHUNK = 100000

# --- Read Metadata and Split Files ---
# (This part remains the same as in the previous consolidated script)
print("Reading metadata...")
try:
    df = pd.read_csv(METADATA_FILE)
except FileNotFoundError:
    print(f"❌ Error: Metadata file not found at {METADATA_FILE}")
    exit()

dev_files = df[df["split"] == "dev"]["path"].tolist()
test_files = df[df["split"] == "test"]["path"].tolist()

if not dev_files:
    print("⚠ Warning: No 'dev' files found in metadata.")
    train_files, val_files = [], []
elif len(dev_files) < 2: # Added check for split feasibility
    print("⚠ Warning: Less than 2 'dev' files found. Cannot perform 80/20 split.")
    train_files = dev_files
    val_files = []
else:
    train_files, val_files = train_test_split(dev_files, test_size=0.2, random_state=42)

print(f"Total LAS files found: Train: {len(train_files)}, Validation: {len(val_files)}, Test: {len(test_files)}")
print(f"Target points per HDF5 chunk file: {POINTS_PER_HDF5_CHUNK}")
print(f"Unclassified label ID to filter: {UNCLASSIFIED_LABEL_ID}")
print(f"Base output directory for chunks: {OUTPUT_ROOT}")


# --- Define Output Subdirectories within OUTPUT_ROOT ---
# These define the split-level directories:
# OUTPUT_ROOT / train_chunks
# OUTPUT_ROOT / val_chunks
# OUTPUT_ROOT / test_chunks
TRAIN_CHUNK_SUBDIR = "train_chunks"
VAL_CHUNK_SUBDIR = "val_chunks"
TEST_CHUNK_SUBDIR = "test_chunks"

# --- Convert the datasets (Using the modified function) ---
print("\n--- Converting Training Set (By Folder - Nested Output) ---")
# Call the function with output_base_subdir as the split-level directory
convert_las_to_filtered_hdf5_chunks_by_folder(train_files, DATASET_ROOT, os.path.join(OUTPUT_ROOT, TRAIN_CHUNK_SUBDIR), "train", POINTS_PER_HDF5_CHUNK, UNCLASSIFIED_LABEL_ID)

print("\n--- Converting Validation Set (By Folder - Nested Output) ---")
convert_las_to_filtered_hdf5_chunks_by_folder(val_files, DATASET_ROOT, os.path.join(OUTPUT_ROOT, VAL_CHUNK_SUBDIR), "val", POINTS_PER_HDF5_CHUNK, UNCLASSIFIED_LABEL_ID)

print("\n--- Converting Test Set (By Folder - Nested Output) ---")
convert_las_to_filtered_hdf5_chunks_by_folder(test_files, DATASET_ROOT, os.path.join(OUTPUT_ROOT, TEST_CHUNK_SUBDIR), "test", POINTS_PER_HDF5_CHUNK, UNCLASSIFIED_LABEL_ID)

Reading metadata...
Total LAS files found: Train: 44, Validation: 12, Test: 26
Target points per HDF5 chunk file: 100000
Unclassified label ID to filter: 0
Base output directory for chunks: /content/drive/My Drive/FOR-Instance_Dataset/preprocessed_folders

--- Converting Training Set (By Folder - Nested Output) ---
Base output directory for 'train' chunks (By Folder - Nested): /content/drive/My Drive/FOR-Instance_Dataset/preprocessed_folders/train_chunks
Starting conversion for 44 files...
⚠ Warning: File not found or not a .las file: /content/drive/My Drive/FOR-Instance_Dataset/NIBIO2/plot41_annotated.las. Skipping.
⚠ Warning: File not found or not a .las file: /content/drive/My Drive/FOR-Instance_Dataset/NIBIO2/plot33_annotated.las. Skipping.
💾 Saving chunk: Data (100000, 6) to /content/drive/My Drive/FOR-Instance_Dataset/preprocessed_folders/train_chunks/NIBIO/train_chunks/train_NIBIO_chunk_0.h5
💾 Saving chunk: Data (100000, 6) to /content/drive/My Drive/FOR-Instance_Dataset/preproc

In [ ]:
def visualize_chunk_plotly(filepath, color_by='instance', point_size=1.0, max_points=50000):
    """
    Loads and visualizes a point cloud chunk from an HDF5 file using Plotly.

    Args:
        filepath (str): Path to the HDF5 chunk file.
        color_by (str): 'semantic', 'instance', or 'none'.
        point_size (float): Size of the points in the visualization.
        max_points (int): Maximum number of points to plot (for performance).
                         Set to None to plot all points (might be slow).
    """
    if not os.path.exists(filepath):
        print(f"Error: File not found at {filepath}")
        return

    print(f"Loading data from: {filepath}")
    try:
        with h5py.File(filepath, 'r') as f:
            if 'data' not in f:
                print("Error: 'data' dataset not found.")
                return
            points = f['data'][:]
            semantic_labels = f['label'][:] if 'label' in f else None
            instance_labels = f['treeID'][:] if 'treeID' in f else None

            if points.shape[0] == 0:
                 print("Warning: Chunk contains 0 points.")
                 return

            n_points = points.shape[0]
            print(f"  Loaded {n_points} points.")

            # --- Subsample if too many points ---
            if max_points is not None and n_points > max_points:
                print(f"  Subsampling to {max_points} points for visualization...")
                indices = np.random.choice(n_points, max_points, replace=False)
                points = points[indices]
                if semantic_labels is not None:
                    semantic_labels = semantic_labels[indices]
                if instance_labels is not None:
                    instance_labels = instance_labels[indices]
                n_points = max_points

            x, y, z = points[:, 0], points[:, 1], points[:, 2]

            # --- Get Colors ---
            colors = None
            color_axis_title = None
            colorscale_name = None # To store colorscale for numeric data

            if color_by == 'semantic' and semantic_labels is not None:
                print("Coloring by Semantic Labels...")
                colors = semantic_labels
                color_axis_title = "Semantic Label"
                colorscale_name = 'Viridis' # Example discrete-like scale
            elif color_by == 'instance' and instance_labels is not None:
                print("Coloring by Instance (TreeID) Labels...")
                colors = instance_labels
                color_axis_title = "Instance ID (TreeID)"
                colorscale_name = 'Turbo' # Good for many distinct values
            elif points.shape[1] >= 6 and color_by == 'rgb':
                print("Coloring by RGB (cols 3-5)...")
                rgb = points[:, 3:6]
                if np.max(rgb) > 1.0: rgb = rgb / 255.0
                # Convert RGB float (0-1) to hex string for plotly
                colors = [f'rgb({int(r*255)},{int(g*255)},{int(b*255)})' for r, g, b in rgb]
                color_axis_title = "RGB"
            else:
                print("No suitable labels/colors found or requested. Using default.")
                colors = 'grey' # Single color

            # --- Create Plotly Figure ---
            fig = go.Figure(data=[go.Scatter3d(
                x=x, y=y, z=z,
                mode='markers',
                marker=dict(
                    size=point_size,
                    color=colors,
                    colorscale=colorscale_name, # Use specified colorscale
                    colorbar=dict(title=color_axis_title) if color_axis_title and colorscale_name else None, # Show colorbar for numeric scales
                    opacity=0.8
                )
            )])

            # Tight layout, set scene aspect ratio
            fig.update_layout(
                title=f'Chunk: {os.path.basename(filepath)} - Colored by: {color_by}',
                margin=dict(l=0, r=0, b=0, t=40), # Add top margin for title
                scene=dict(aspectmode='data')
            )
            print("Displaying interactive plot...")
            fig.show()

    except Exception as e:
        print(f"An error occurred: {e}")
        import traceback
        traceback.print_exc()

In [ ]:
# --- Specify your file path ---
chunk_file_path = '/content/drive/MyDrive/FOR-Instance_Dataset/preprocessed_chunks/val_chunks/val_chunk_90.h5'

In [ ]:
# --- Visualize ---
print("--- Visualizing by Instance ID ---")
visualize_chunk_plotly(chunk_file_path, color_by='instance', point_size=1, max_points=75000) # Limit points for performance

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
print("\n--- Visualizing by Semantic Label ---")
visualize_chunk_plotly(chunk_file_path, color_by='semantic', point_size=1, max_points=75000)

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
# --- Specify your file path ---
chunk_file_path = '/content/drive/MyDrive/FOR-Instance_Dataset/preprocessed_chunks/val_chunks/val_chunk_90.h5'